In [2]:
import os
import json
import pandas as pd
import torch

In [3]:
with open("/home/bbc8731/NetMedGPT/data/parameters.json", 'r') as file:
    all_param = json.load(file)

data_dir = all_param['files']['data_dir']

In [4]:
nodes = pd.read_csv(os.path.join(data_dir, 'nodes.csv'), sep= ',')
edge = pd.read_csv(os.path.join(data_dir, "edges.csv")) 

In [7]:
# for loading the model
mask_token = edge['z_index'].max() + 1
vocab_size = mask_token + 1

relation_type = list(edge['relation'].unique())
node_types = list(nodes['node_type'].unique())
node_types_without_feat = ['biological_process',
                           'molecular_function',
                           'cellular_component',
                           'exposure',
                           'pathway',
                           'anatomy']

mask = ['mask']
entity = node_types + relation_type + mask

without_feat = {
    'node': node_types_without_feat,
    'relation': relation_type,
    'mask': mask
}

## find relation_mask_index
relation_index = edge.loc[edge['relation'].isin(relation_type), ['relation', 'z_index']].drop_duplicates()
mask_row = pd.DataFrame([['mask', mask_token]], columns=['relation', 'z_index'])
relation_mask_index = pd.concat([relation_index, mask_row], ignore_index=True)

# Generate hyperparameter combinations
param_grid = {
    'hidden_channels': [300],    
    'walk_length': [5],         
    'walks_per_node': [30],      
    'nhead': [10],               
    'N_encoder_layers': [20],     
    'batch_size': [4000],       
    'learning_rate': [0.0001]    
}

drug_pubmedbert = pd.read_csv(os.path.join(data_dir, 'attr_drug_pubmedbert.csv'), sep= ',', index_col=0)
drug_pubmedbert = drug_pubmedbert.sort_index()
drug_pubmedbert = torch.tensor(drug_pubmedbert.values, dtype=torch.float32)  

drug_FP = pd.read_csv(os.path.join(data_dir, 'attr_drug_fingerprint.csv'), sep= ',')
drug_FP = drug_FP.sort_index()
drug_FP = torch.tensor(drug_FP.values, dtype=torch.float32)  

gene_pubmedbert = pd.read_csv(os.path.join(data_dir, 'attr_protein_pubmedbert.csv'), sep= ',', index_col=0)
gene_pubmedbert = gene_pubmedbert.sort_index()
gene_pubmedbert = torch.tensor(gene_pubmedbert.values, dtype=torch.float32)  

gene_esm2 = torch.load(os.path.join(data_dir, 'attr_protein_emb_esm2.pt'))

disease_pubmedbert = pd.read_csv(os.path.join(data_dir, 'attr_disease_pubmedbert.csv'), sep= ',', index_col = 0)
disease_pubmedbert = disease_pubmedbert.sort_index()
disease_pubmedbert = torch.tensor(disease_pubmedbert.values, dtype=torch.float32)  

phenotype_pubmedbert = pd.read_csv(os.path.join(data_dir, 'attr_phenotype_pubmedbert.csv'), sep= ',', index_col = 0)
phenotype_pubmedbert = phenotype_pubmedbert.sort_index()
phenotype_pubmedbert = torch.tensor(phenotype_pubmedbert.values, dtype=torch.float32)  

# make a dict of all the features
feat = {
    'gene/protein': {
        'pubmedbert': gene_pubmedbert,
        'esm2': gene_esm2
    },
    'drug': {
        'pubmedbert': drug_pubmedbert,
        'FP': drug_FP
    },
    'disease': {
        'pubmedbert': disease_pubmedbert,
    },
    'effect/phenotype': {
        'pubmedbert': phenotype_pubmedbert,
    }
}

feat_rest = {}
for k, v in without_feat.items():
    if k == 'node':
        for i in v:
            N_nodes_i = (nodes['node_type'] == i).sum()
            feat_i = torch.randn(N_nodes_i, param_grid['hidden_channels'][0])
            feat_rest[f"{i}"] = {'random': feat_i}

    elif k == 'relation':
         for i in v:
             feat_i = torch.ones(1, param_grid['hidden_channels'][0])
             feat_rest[f"{i}"] = {'fixed': feat_i}   # I should put this also fixed instead of random, but now the result is based on random
    elif k == 'mask':
        feat_i = torch.ones(1,param_grid['hidden_channels'][0])
        feat_rest[f"{k}"] = {'fixed': feat_i}   

feat.update(feat_rest)


In [8]:
torch.save(feat, os.path.join(data_dir, "embeddings_with_feat.pt"))
